In [1]:
import os
import sys
from ast import literal_eval
import pandas as pd
import spacy
import re
from spacy.tokenizer import Tokenizer
import matplotlib.pyplot as plt
import dill
import numpy as np

# Add intent_recognition to sys.path so `from src.services...` imports resolve
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(''), '..', '..', '..', 'intent_recognition')))

In [2]:
data = pd.read_csv('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv')

In [3]:
data = data[data['is_pfub']==True].reset_index(drop=True)

In [4]:
data

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice
0,41c5ecfd-2b9d-5364-81b8-db9d86577ea5,16dd90f1-4ed1-5dd1-98d5-1408bc44b052,Amtsgericht Memmingen\nAbteilung für Zwangsvol...,01.04.2025/ocr-v2_54_M_1067_25_Schreiben_vom_3...,approved_attachment_and_transfer_order,amtsgericht memmingen\nabteilung für zwangsvol...,"{'case_slug': '163464253141', 'debtor_name': '...",True,False,NaN,d3220801ad866b43be8d71cebdd044cd9cbedae7ff5be4...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN
1,737e68b1-d955-5abd-89a8-3c9bcfd89ebd,350b91d6-2dfd-5be6-bd1d-58a4c9eacf02,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,01.06.2022/1654077659_Doc_01062022_000000001_1...,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '110872558124', 'debtor_name': '...",True,False,NaN,39f6bde0d7784c8fa3429f3108940f87d90d22c690f63f...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN
2,f79cd1b4-9650-574a-935b-ddd715c6c4ca,37945cc0-5e4a-59ce-b00e-aea9b53d4368,Amtsgericht Eschweiler\n070071 967072\n-Geschä...,01.06.2022/1654077662_Doc_01062022_000000001_1...,approved_seizure,amtsgericht eschweiler\n070071 967072\n-geschä...,"{'case_slug': '114249164962', 'debtor_name': '...",True,False,NaN,1839b0679e88df1c3695e07adc919edf69e15c20b24924...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN
3,2116361b-c0e3-55bf-8f20-5f40d00525ce,7df7ef68-9412-5f70-bd3d-9a6d761f7223,Amtsgericht Münster\n-33- Amtsgericht Münster ...,01.07.2024/ocr-v2_BB_005_01072024_122825.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '136916228754', 'debtor_name': '...",True,False,NaN,8b9ea31725e645286d54e60bab67292043350aaf8f2814...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN
4,42e06686-1997-50ad-9663-49ce18779b46,001657d6-e0ee-5a70-90cb-55e74e486d35,Amtsgericht Viersen\n-Geschäftsstelle-\n-15- A...,01.07.2025/ocr-v2_Allgemeines_Schreiben_R__202...,approved_attachment_and_transfer_order,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,"{'case_slug': '125495503778', 'debtor_name': '...",True,False,NaN,663be49a05043aa36dd9bc1cbbc279d48a087447040b73...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
466,2e376cd6-1c71-5d2c-9e3d-b0e2de8d7d53,53c634ab-3f82-5a9c-8b5d-6cf03a6313da,Amtsgericht Wuppertal\n-44- Amtsgericht Wupper...,31.01.2023/1675166710_YA_003_31012023_124814.pdf,approved_seizure,amtsgericht wuppertal\n-44- amtsgericht wupper...,"{'case_slug': '110454161073', 'debtor_name': '...",True,False,NaN,519fef65d2e57ee9d84188b1aae63288090cd1b5b6f36a...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN
467,b165ce10-36d3-53fc-a6d4-9be48a4d3cc4,0877e12f-f36c-5761-ac9e-141fccf57eba,"Amtsgericht Itzehoe\nAmtsgericht Itzehoe, Berg...",31.01.2023/1675166710_YA_003_31012023_124815.pdf,approved_seizure,"amtsgericht itzehoe\namtsgericht itzehoe, berg...","{'case_slug': None, 'debtor_name': None, 'cred...",True,False,NaN,2542df394fd597979716f57116e03582c6d7026c136b3f...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN
468,421731ab-24dc-5e82-b1eb-940cb53ac48e,e7370a9d-5400-5d84-bdfe-a82af0cb7dfb,Amtsgericht Leverkusen\n-45- Amtsgericht Lever...,31.05.2023/1685534803_PF_005_31052023_140508.pdf,approved_seizure,amtsgericht leverkusen\n-45- amtsgericht lever...,"{'case_slug': '117578968598', 'debtor_name': '...",True,False,NaN,b76f1bb4fd8e817aab997c109fe5c7a13e3aca72c058a9...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN
469,31d2513b-9ad1-53fc-a31f-c91f7e846ff2,26f76c76-fc0d-5109-9f45-427fc1a24bc9,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,31.07.2023/1690794316_KD_004_31072023_110320.pdf,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '125677855012', 'debtor_name': '...",True,False,NaN,db30fc9d20ccd495194936c243c753fe06765697d4e620...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN


In [5]:
def extract_data(data):
    try:
        data = literal_eval(data)
        case_slug = data['case_slug'] if 'case_slug' in data else None
        creditor_name = data['creditor_name'] if 'creditor_name' in data else None
    except:
        case_slug = None
        creditor_name = None
    return case_slug, creditor_name

data[['case_slug','creditor_name']] = data['data'].apply(lambda x: pd.Series(extract_data(x)))

In [6]:
data.dropna(subset=['creditor_name'], inplace=True)

In [7]:
data

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,case_slug,creditor_name
0,41c5ecfd-2b9d-5364-81b8-db9d86577ea5,16dd90f1-4ed1-5dd1-98d5-1408bc44b052,Amtsgericht Memmingen\nAbteilung für Zwangsvol...,01.04.2025/ocr-v2_54_M_1067_25_Schreiben_vom_3...,approved_attachment_and_transfer_order,amtsgericht memmingen\nabteilung für zwangsvol...,"{'case_slug': '163464253141', 'debtor_name': '...",True,False,NaN,d3220801ad866b43be8d71cebdd044cd9cbedae7ff5be4...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,163464253141,Liquandum Capital GmbH
1,737e68b1-d955-5abd-89a8-3c9bcfd89ebd,350b91d6-2dfd-5be6-bd1d-58a4c9eacf02,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,01.06.2022/1654077659_Doc_01062022_000000001_1...,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '110872558124', 'debtor_name': '...",True,False,NaN,39f6bde0d7784c8fa3429f3108940f87d90d22c690f63f...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,110872558124,Immobilien Scout GmbH
2,f79cd1b4-9650-574a-935b-ddd715c6c4ca,37945cc0-5e4a-59ce-b00e-aea9b53d4368,Amtsgericht Eschweiler\n070071 967072\n-Geschä...,01.06.2022/1654077662_Doc_01062022_000000001_1...,approved_seizure,amtsgericht eschweiler\n070071 967072\n-geschä...,"{'case_slug': '114249164962', 'debtor_name': '...",True,False,NaN,1839b0679e88df1c3695e07adc919edf69e15c20b24924...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,114249164962,OVAG Ostdeutsche Versicherung AG
3,2116361b-c0e3-55bf-8f20-5f40d00525ce,7df7ef68-9412-5f70-bd3d-9a6d761f7223,Amtsgericht Münster\n-33- Amtsgericht Münster ...,01.07.2024/ocr-v2_BB_005_01072024_122825.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '136916228754', 'debtor_name': '...",True,False,NaN,8b9ea31725e645286d54e60bab67292043350aaf8f2814...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,136916228754,Liquandum Capital GmbH
4,42e06686-1997-50ad-9663-49ce18779b46,001657d6-e0ee-5a70-90cb-55e74e486d35,Amtsgericht Viersen\n-Geschäftsstelle-\n-15- A...,01.07.2025/ocr-v2_Allgemeines_Schreiben_R__202...,approved_attachment_and_transfer_order,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,"{'case_slug': '125495503778', 'debtor_name': '...",True,False,NaN,663be49a05043aa36dd9bc1cbbc279d48a087447040b73...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,125495503778,Grover Group GmbH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
464,0bb61b0b-9582-50f6-a24b-ba449b671aab,6bf504fb-5358-5c6a-bafe-f624745da00c,Amtsgericht Heinsberg\n4070071967072\n-10- Amt...,30.08.2022/1661853645_Scan_YA_00330082022_1141...,approved_seizure,amtsgericht heinsberg\n4070071967072\n-10- amt...,"{'case_slug': '107672088773', 'debtor_name': '...",True,False,NaN,1ac9d4b14b3f71c17f0457a8359c8e5808a66162794026...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,107672088773,Liquandum Capital GmbH
465,92385e88-1264-54ba-97b2-3a71221526c4,8b4b44c7-6fa8-5ff4-9d25-0eba6cc2e846,Amtsgericht Münster\n-33- Amtsgericht Münster ...,30.10.2025/ocr-v2_c0872fd7aa1c431e.pdf,approved_attachment_and_transfer_order,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '181060314323', 'debtor_name': '...",True,False,NaN,2e2ace9d7d5a381975ca3541f0a81a3e8d161f2c6686c1...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,181060314323,Liquandum Capital II GmbH
466,2e376cd6-1c71-5d2c-9e3d-b0e2de8d7d53,53c634ab-3f82-5a9c-8b5d-6cf03a6313da,Amtsgericht Wuppertal\n-44- Amtsgericht Wupper...,31.01.2023/1675166710_YA_003_31012023_124814.pdf,approved_seizure,amtsgericht wuppertal\n-44- amtsgericht wupper...,"{'case_slug': '110454161073', 'debtor_name': '...",True,False,NaN,519fef65d2e57ee9d84188b1aae63288090cd1b5b6f36a...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,110454161073,freenet DLS GmbH
468,421731ab-24dc-5e82-b1eb-940cb53ac48e,e7370a9d-5400-5d84-bdfe-a82af0cb7dfb,Amtsgericht Leverkusen\n-

In [8]:

import re
from collections import defaultdict
from typing import Optional, List
import spacy
from spacy.matcher import Matcher

from src.services.attachment_processing.aftercourt_extractors.base import BaseExtractor
from src.services.attachment_processing.aftercourt_extractors.base import NLPModelManager


class CreditorNameExtractorNewPfub(BaseExtractor):
    def __init__(self, nlp_model):
        """
        Initialize the creditor name extractor
        
        Args:
            nlp_model: spaCy language model (e.g., spacy.load("de_core_news_sm"))
        """
        self.nlp = nlp_model
        self.gegen_matcher = Matcher(self.nlp.vocab)
        pattern = [{"LOWER": "gegen"}]
        # Add pattern to match "./." as well
        pattern_dot_slash = [{"LOWER": "./."}]
        self.gegen_matcher.add("DOT_SLASH_PATTERN", [pattern_dot_slash])
        self.gegen_matcher.add("GEGEN_PATTERN", [pattern])
    
    def extract(self, text: str) -> Optional[str]:
        """Extract creditor name from text"""
        if not text:
            return None
        
        # Collect context windows around 'gegen' matches
        context_window = self._collect_context_windows(text)
        # Extract creditor name
        creditor_name = self._extract_validated_creditor_name(context_window) if context_window else None
        
        # Name and Surname is capitalized
        if creditor_name:
            creditor_name = ' '.join(word.capitalize() for word in creditor_name.split())

        return creditor_name
    
    def _is_valid_context(self, left_ctx, right_ctx) -> bool:
        """Check if context contains valid entities (PER/LOC/ORG)"""
        left_entities_labels = [ent.label_ for ent in left_ctx.ents]
        right_entities_labels = [ent.label_ for ent in right_ctx.ents]

        if not (any(label in left_entities_labels for label in ["PER", "LOC", "ORG"]) or 
                any(label in right_entities_labels for label in ["PER", "LOC", "ORG"])):
            return False

        return True

    def _get_context_window(self, doc, start, end, window=10) -> Optional[spacy.tokens.span.Span]:
        """Extract context window around the 'gegen' match. Since we want to extract creditor name, get the left context"""
        left_start = max(start - window, 0)
        right_end = min(end + window, len(doc))
        left_ctx = doc[left_start:start]
        right_ctx = doc[end:right_end]
        flag = self._is_valid_context(left_ctx, right_ctx) # for now dont care for if its valid context
        return left_ctx

    def _collect_context_windows(self, text: str) -> Optional[List[spacy.tokens.span.Span]]:
        """
        In pdub documents, the classic pattern is <CREDITOR NAME> 'gegen or './.' <DEBTOR NAME>.
        Therefore this function collects context windows around 'gegen' or './.' matches and returns the first valid one.
        The creditor name is then extracted from this context window.
        Args:
            text (str): Input text to extract context windows from
        Returns:
            Optional[List[spacy.tokens.span.Span]]: List of valid context windows or None if none found
        """
        doc = self.nlp(text)
    
        gegen_matches = self.gegen_matcher(doc)
        if not gegen_matches:
            return None
        context_windows = []
        for _, start, end in gegen_matches:
            context = self._get_context_window(doc, start, end, window=15)
            if context:
                context_windows.append(context)
                
        if not context_windows:
            return None

        return context_windows[0]  # Return only the first valid context window

    def _find_used_delimiter(self, context) -> Optional[str]:
        """Find the most frequently used delimiter in the context"""
        used_delimiter = defaultdict(int)
        for t in context:
            if t.is_punct and not t.is_space:
                used_delimiter[t.text] += 1
        # get the most frequent
        most_frequent = max(used_delimiter, key=used_delimiter.get, default=None)
        return most_frequent

    def _validate_creditor_name(self, creditor_name: str) -> Optional[str]:
        """Clean and validate the extracted creditor name"""
        if not creditor_name:
            return None
            
        creditor_name = creditor_name.lower()
        creditor_name = re.sub(r'\b(herrn?|frau|fräulein)\b', '', creditor_name)
        creditor_name = creditor_name.replace('c/o', '')
        # remove 'zwangsvollstreckungssache' if its present in the creditor name
        creditor_name = re.sub(r'\bzwangsvollstreckungssache\b', '', creditor_name, flags=re.IGNORECASE)
        # Remove numbers and punctuation, keep only letters, spaces, and 'ß'
        creditor_name = re.sub(r'[^a-zA-ZÀ-ÿß\s-]', '', creditor_name)
        creditor_name = creditor_name.lstrip()
        # remove after \n detected
        creditor_name = re.sub(r'\n.*', '', creditor_name)
        # Clean up multiple spaces and strip
        creditor_name = re.sub(r'\s+', ' ', creditor_name).strip()
        return creditor_name

    def _contains_salutation(self, text: str) -> bool:
        """Check if text contains German salutations"""
        salutation_patterns = [
            r'\b(herrn?|frau|fräulein)\b',
        ]
        return any(re.search(pattern, text, re.IGNORECASE) for pattern in salutation_patterns)

    def _extract_validated_creditor_name(self, context_window) -> Optional[str]:
        """Extract creditor name from context window"""
        if not context_window:
            return None
        
        creditor_name: str = None
        blocks_by_line_break = context_window.text.split("\n")
        
        if len(blocks_by_line_break)>1:
            last_block = blocks_by_line_break[-1].strip()
            last_second_block = blocks_by_line_break[-2].strip()
            if len(last_block)>1:
                creditor_name = last_block
                creditor_name = self._validate_creditor_name(creditor_name)

        return creditor_name

In [9]:
new_extractor = CreditorNameExtractorNewPfub(NLPModelManager.get_model("de_core_news_md"))

In [10]:
new_extractor.extract(data.iloc[0]['cleaned_text'])

'Liquandum Capital Ii Gmbh'

In [11]:
data.iloc[0]

ticket_uuid                        41c5ecfd-2b9d-5364-81b8-db9d86577ea5
attachment_id                      16dd90f1-4ed1-5dd1-98d5-1408bc44b052
text                  Amtsgericht Memmingen\nAbteilung für Zwangsvol...
object_key            01.04.2025/ocr-v2_54_M_1067_25_Schreiben_vom_3...
document_type                    approved_attachment_and_transfer_order
cleaned_text          amtsgericht memmingen\nabteilung für zwangsvol...
data                  {'case_slug': '163464253141', 'debtor_name': '...
is_pfub                                                            True
is_ladung                                                         False
s3_link                                                             NaN
textract_job_id       d3220801ad866b43be8d71cebdd044cd9cbedae7ff5be4...
textract_s3_link      s3://pair-data-engineering-new/ocr_prepared_ou...
is_ve_with_invoice                                                  NaN
case_slug                                                  16346

In [12]:
data['context_window'] = data['cleaned_text'].apply(lambda x: new_extractor._collect_context_windows(x))

In [13]:
data

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,case_slug,creditor_name,context_window
0,41c5ecfd-2b9d-5364-81b8-db9d86577ea5,16dd90f1-4ed1-5dd1-98d5-1408bc44b052,Amtsgericht Memmingen\nAbteilung für Zwangsvol...,01.04.2025/ocr-v2_54_M_1067_25_Schreiben_vom_3...,approved_attachment_and_transfer_order,amtsgericht memmingen\nabteilung für zwangsvol...,"{'case_slug': '163464253141', 'debtor_name': '...",True,False,NaN,d3220801ad866b43be8d71cebdd044cd9cbedae7ff5be4...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,163464253141,Liquandum Capital GmbH,"(54, m, 1067, /, 25, \n, 31.03.2025, \n, in, s..."
1,737e68b1-d955-5abd-89a8-3c9bcfd89ebd,350b91d6-2dfd-5be6-bd1d-58a4c9eacf02,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,01.06.2022/1654077659_Doc_01062022_000000001_1...,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '110872558124', 'debtor_name': '...",True,False,NaN,39f6bde0d7784c8fa3429f3108940f87d90d22c690f63f...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,110872558124,Immobilien Scout GmbH,"(\n, sehr, geehrte, damen, und, herren, ,, \n,..."
2,f79cd1b4-9650-574a-935b-ddd715c6c4ca,37945cc0-5e4a-59ce-b00e-aea9b53d4368,Amtsgericht Eschweiler\n070071 967072\n-Geschä...,01.06.2022/1654077662_Doc_01062022_000000001_1...,approved_seizure,amtsgericht eschweiler\n070071 967072\n-geschä...,"{'case_slug': '114249164962', 'debtor_name': '...",True,False,NaN,1839b0679e88df1c3695e07adc919edf69e15c20b24924...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,114249164962,OVAG Ostdeutsche Versicherung AG,"(sehr, geehrte, damen, und, herren, ,, \n, in,..."
3,2116361b-c0e3-55bf-8f20-5f40d00525ce,7df7ef68-9412-5f70-bd3d-9a6d761f7223,Amtsgericht Münster\n-33- Amtsgericht Münster ...,01.07.2024/ocr-v2_BB_005_01072024_122825.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '136916228754', 'debtor_name': '...",True,False,NaN,8b9ea31725e645286d54e60bab67292043350aaf8f2814...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,136916228754,Liquandum Capital GmbH,None
4,42e06686-1997-50ad-9663-49ce18779b46,001657d6-e0ee-5a70-90cb-55e74e486d35,Amtsgericht Viersen\n-Geschäftsstelle-\n-15- A...,01.07.2025/ocr-v2_Allgemeines_Schreiben_R__202...,approved_attachment_and_transfer_order,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,"{'case_slug': '125495503778', 'debtor_name': '...",True,False,NaN,663be49a05043aa36dd9bc1cbbc279d48a087447040b73...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,125495503778,Grover Group GmbH,"(\n, sehr, geehrte, damen, und, herren, ,, \n,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
464,0bb61b0b-9582-50f6-a24b-ba449b671aab,6bf504fb-5358-5c6a-bafe-f624745da00c,Amtsgericht Heinsberg\n4070071967072\n-10- Amt...,30.08.2022/1661853645_Scan_YA_00330082022_1141...,approved_seizure,amtsgericht heinsberg\n4070071967072\n-10- amt...,"{'case_slug': '107672088773', 'debtor_name': '...",True,False,NaN,1ac9d4b14b3f71c17f0457a8359c8e5808a66162794026...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,107672088773,Liquandum Capital GmbH,"(\n, sehr, geehrte, damen, und, herren, ,, \n,..."
465,92385e88-1264-54ba-97b2-3a71221526c4,8b4b44c7-6fa8-5ff4-9d25-0eba6cc2e846,Amtsgericht Münster\n-33- Amtsgericht Münster ...,30.10.2025/ocr-v2_c0872fd7aa1c431e.pdf,approved_attachment_and_transfer_order,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '181060314323', 'debtor_name': '...",True,False,NaN,2e2ace9d7d5a381975ca3541f0a81a3e8d161f2c6686c1...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,181060314323,Liquandum Capital II GmbH,None
466,2e376cd6-1c71-5d2c-9e3d-b0e2de8d7d53,53c634ab-3f82-5a9c-8b5d-6cf03a6313da,Amtsgericht Wuppertal\n-44- Amtsgericht Wupper...,31.01.2023/1675166710_YA_003_31012023_124814.pdf,approved_seizure,amtsgericht wuppertal\n-44- amtsgericht wupper...,"{'case_slug': 

## Check context windows to have general idea

In [14]:
# check context_window = None
context_window_none = data[data['context_window'].isnull()]

In [15]:
context_window_none

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link,is_ve_with_invoice,case_slug,creditor_name,context_window
3,2116361b-c0e3-55bf-8f20-5f40d00525ce,7df7ef68-9412-5f70-bd3d-9a6d761f7223,Amtsgericht Münster\n-33- Amtsgericht Münster ...,01.07.2024/ocr-v2_BB_005_01072024_122825.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '136916228754', 'debtor_name': '...",True,False,NaN,8b9ea31725e645286d54e60bab67292043350aaf8f2814...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,136916228754,Liquandum Capital GmbH,None
10,76363253-1415-5aa3-85d7-6ed0d5fcb517,ae1f84d9-bef4-58ba-803e-3b9f3f082960,"Amtsgericht Münster\nHaws "", asueula PAIR\nJul...",01.08.2025/ocr-v2_KL_006-01082025-080649.pdf,approved_seizure,"amtsgericht münster\nhaws "", asueula pair\njul...","{'case_slug': '151012935186', 'debtor_name': '...",True,False,NaN,0f4be4caab1bbd40d3291eed1bb8f449e54494f4f17c1c...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,151012935186,Jochen Schweizer,None
25,17094d6d-6fae-570f-b44b-2f7c23305094,a6d8dc09-949a-5164-b988-05d4cac77577,Amtsgericht Münster\n-33- Amtsgericht Münster ...,02.07.2025/ocr-v2_KL_005-02072025-085434.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '166850524785', 'debtor_name': '...",True,False,NaN,f1cc46625f823c575c50391873e148730d902666412756...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,166850524785,Liquandum Capital GmbH,None
31,db942fb7-a775-5816-99ec-c71af946efc5,179e07e6-24cf-5b50-a6e2-51b93d36f127,Amtsgericht Münster\n-33- Amtsgericht Münster ...,02.10.2023/1696236517_KD_001_02102023_103939.pdf,approved_attachment_and_transfer_order,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '111380937636', 'debtor_name': '...",True,False,NaN,8e4ca24bd0c668b36780796e4b174cbf50bec5ba95fe1c...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,111380937636,111380937636,None
40,00d445c1-7be2-5e2b-bc21-641f7c48ff98,f5af7b2f-0f5c-556a-962c-5af9c655e60c,Amtsgericht Münster\n-33- Amtsgericht Münster ...,04.02.2025/ocr-v2_FA_009_04022025_132655.pdf,approved_attachment_and_transfer_order,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '140879090003', 'debtor_name': '...",True,False,NaN,bc817c0b0d0ccff1d62e098964c3b50b3209cc8c96b35d...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,140879090003,Liquandum Capital GmbH,None
41,224d6536-9c49-5e0d-a918-36a6ad7bc8b9,22748192-a0f5-5c97-b563-834b8da7a0a4,Amtsgericht Münster\n-33- Amtsgericht Münster ...,04.03.2025/ocr-v2_KL_001-04032025-121228.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '157242486957', 'debtor_name': '...",True,False,NaN,96b764e06fef42b093fab3873ea9ab7787ea67594ab1a1...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,157242486957,Mondu,None
55,8edd35d0-32fe-58ea-9398-845fd7db44d5,7a61407c-5dde-5723-a489-03d97c55cf34,Amtsgericht Münster\n-33- Amtsgericht Münster ...,04.09.2025/ocr-v2_FA_008-04092025-081100.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '129328972564', 'debtor_name': '...",True,False,NaN,e2945facb256583deab417bfd376f27297d4fd22c7c6bb...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,129328972564,QCells,None
56,995f342e-9bff-51a9-a8ec-3af0a83c2cae,ead02b14-6edf-5a61-801c-623258726730,Amtsgericht Münster\n-33- Amtsgericht Münster ...,04.09.2025/ocr-v2_b655aab1cf7f435d.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '166930537179', 'debtor_name': '...",True,False,NaN,6723585ec50d06d269270a945288e4cf61950c08553e66...,s3://pair-data-engineering-new/ocr_prepared_ou...,NaN,166930537179,Liquandum Capital GmbH,None
114,45d0bc26-faaf-58b2-a6bb-15880508a914,36faf893-167a-5675-831b-c25a2d21c5cd,Amtsgericht Münster\n-33- Amtsgericht Münster ...,08.08.2025/ocr-v2_RB_001-08082025-080718.pdf,approved_seizure,amtsgericht 

In [16]:
def print_out_clickable_link(df):
    df = df.reset_index()
    for index, row in df.iterrows():
        print(f"Index: {index}, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix={row['object_key']}")

In [17]:
print_out_clickable_link(context_window_none)

Index: 0, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=01.07.2024/ocr-v2_BB_005_01072024_122825.pdf
Index: 1, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=01.08.2025/ocr-v2_KL_006-01082025-080649.pdf
Index: 2, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=02.07.2025/ocr-v2_KL_005-02072025-085434.pdf
Index: 3, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=02.10.2023/1696236517_KD_001_02102023_103939.pdf
Index: 4, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=04.02.2025/ocr-v2_FA_009_04022025_132655.pdf
Index: 5, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=04.03.2025/ocr-v2_KL_001-04032025-121228.pdf
Index: 6, Link: https://eu-central-1.console.aws.amazon.com/s3/obj

## Result: CHecked all 25 data, there is no creditor name in them. So theye are fine.

In [18]:
# check context window
check = data[data['context_window'].notnull()][['context_window','creditor_name']]
check['context_window_text'] = check['context_window'].apply(lambda x: repr(x.text) if x else None)
check['context_window_text'] = check['context_window_text'].apply(lambda x: x.replace('\\n', '<LINE_BREAK>'))
check

,context_window,creditor_name,context_window_text
0,"(54, m, 1067, /, 25, \n, 31.03.2025, \n, in, s...",Liquandum Capital GmbH,'54 m 1067/25<LINE_BREAK>31.03.2025<LINE_BREAK...
1,"(\n, sehr, geehrte, damen, und, herren, ,, \n,...",Immobilien Scout GmbH,"'<LINE_BREAK>sehr geehrte damen und herren,<LI..."
2,"(sehr, geehrte, damen, und, herren, ,, \n, in,...",OVAG Ostdeutsche Versicherung AG,"'sehr geehrte damen und herren,<LINE_BREAK>in ..."
4,"(\n, sehr, geehrte, damen, und, herren, ,, \n,...",Grover Group GmbH,"'<LINE_BREAK>sehr geehrte damen und herren,<LI..."
5,"(sehr, geehrte, damen, und, herren, ,, \n, in,...",disapo.de Apotheke B.V,"'sehr geehrte damen und herren,<LINE_BREAK>in ..."
...,...,...,...
463,"(\n, sehr, geehrte, damen, und, herren, ,, \n,...",Jochen Schweizer GmbH,"'<LINE_BREAK>sehr geehrte damen und herren,<LI..."
464,"(\n, sehr, geehrte, damen, und, herren, ,, \n,...",Liquandum Capital GmbH,"'<LINE_BREAK>sehr geehrte damen und herren,<LI..."
466,"(\n, sehr, geehrte, damen, und, herren, ,, \n,...",freenet DLS GmbH,"'<LINE_BREAK>sehr geehrte damen und herren,<LI..."
468,"(\n, sehr, geehrte, damen, und, herren, ,, \n,...",BavariaDirekt Versicherung AG,"'<LINE_BREAK>sehr geehrte damen und herren,<LI..."


In [20]:
data['extracted_creditor_name'] = data.apply(lambda row: new_extractor.extract(row['cleaned_text']), axis=1)

In [21]:
check = data[['cleaned_text','creditor_name','extracted_creditor_name','context_window']].copy()
check.dropna(subset=['context_window'], inplace=True)
check.reset_index(inplace=True, drop=True)
check

,cleaned_text,creditor_name,extracted_creditor_name,context_window
0,amtsgericht memmingen\nabteilung für zwangsvol...,Liquandum Capital GmbH,Liquandum Capital Ii Gmbh,"(54, m, 1067, /, 25, \n, 31.03.2025, \n, in, s..."
1,amtsgericht frankfurt am main\nmobiliarzwangsv...,Immobilien Scout GmbH,Immobilien Scout Gmbh,"(\n, sehr, geehrte, damen, und, herren, ,, \n,..."
2,amtsgericht eschweiler\n070071 967072\n-geschä...,OVAG Ostdeutsche Versicherung AG,Ovag Ostdeutsche Versicherung Ag,"(sehr, geehrte, damen, und, herren, ,, \n, in,..."
3,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,Grover Group GmbH,Grover Group Gmbh,"(\n, sehr, geehrte, damen, und, herren, ,, \n,..."
4,amtsgericht köln\npair finance gmbh\ns c. juni...,disapo.de Apotheke B.V,Disapode Apotheke Bv,"(sehr, geehrte, damen, und, herren, ,, \n, in,..."
...,...,...,...,...
370,"amtsgericht köln\n-288a- amtsgericht köln, 509...",Jochen Schweizer GmbH,Jochen Schweizer Gmbh,"(\n, sehr, geehrte, damen, und, herren, ,, \n,..."
371,amtsgericht heinsberg\n4070071967072\n-10- amt...,Liquandum Capital GmbH,Liquandum Capital Gmbh,"(\n, sehr, geehrte, damen, und, herren, ,, \n,..."
372,amtsgericht wuppertal\n-44- amtsgericht wupper...,freenet DLS GmbH,Freenet Dls Gmbh,"(\n, sehr, geehrte, damen, und, herren, ,, \n,..."
373,amtsgericht leverkusen\n-45- amtsgericht lever...,BavariaDirekt Versicherung AG,Bavariadirekt Versicherung Ag,"(\n, sehr, geehrte, damen, und, herren, ,, \n,..."


In [22]:
wrong = check.iloc[19]
wrong

cleaned_text               pair finance gmbh\n31. mai 2025\namtsgericht l...
creditor_name                                         Liquandum Capital GmbH
extracted_creditor_name                            Liquandum Capital Ii Gmbh
context_window             (bei, antwort, angeben, ), \n, ihr, zeichen, :...
Name: 19, dtype: object

In [23]:
wrong['cleaned_text']

'pair finance gmbh\n31. mai 2025\namtsgericht leipzig\namtsgericht leipzig\nvollstreckungsgericht\nbernhard-göring-straße 64, 04275 leipzig\n446 m 5997/25\nleipzig, 27.05.2025\npair finance gmbh\ngeschäftsstelle\nknesebeckstraße 62-63\ntelefon: 0341 4940 177 (fr philipp)\n10719 berlin\n0341 4940 175 (fr winzler)\ntelefax: 0341 4940 151\naktenzeichen: 446 m 5995/25\n(bitte bei antwort angeben)\nihr zeichen: 159644627667\nzwangsvollstreckungssache liquandum capital ii gmbh ./. kleiner, toby\nsehr geehrte damen und herren,\nunter dem aktenzeichen 446 m 5997/25 wurde heute ein identischer pfändungs- und über-\nweisungsbeschluss erlassen. unter dem hiesigen aktenzeichen wurde der antrag nochmals\neingereicht. ein nochmaliger erlass ist nicht vorgesehen. bitte nehmen sie diesen antrag bin-\nnen 2 wochen zurück.\nmit freundlichen grüßen\nauf anordnung\nhainke\nurkundsbeamtin der geschäftsstelle\nhinweise zum datenschutz erhallen sie auf unserer internetseile auf wunsch senden wir ihnen diese 

In [24]:
result = new_extractor.extract(wrong['cleaned_text'])
print(result)

Liquandum Capital Ii Gmbh
